In [ ]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.version.cuda)  # Should show 11.8 or similar
print(f"Available GPUs: {torch.cuda.device_count()}")
print(f"Current GPU: {torch.cuda.current_device()}")
print(f"GPU Name: {torch.cuda.get_device_name(torch.cuda.current_device())}")


In [ ]:
import os
import torch
from ultralytics import YOLO

# Determine the device to use
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Set paths
base_path = r"X:\Optical_PHOTOMOSAIC_03\AIML\models\AGLO_models\V8\split"
yaml_file_path = os.path.join(base_path, 'data.yaml')

output_path = r"X:\Optical_PHOTOMOSAIC_03\AIML\models\AGLO_models\V8.7"
# Load the smaller YOLO11 model
model = YOLO("yolo11m.pt")

# Move the model to the correct device
model.model.to(device)

# Freeze the first few layers for the first 10 epochs for better fine-tuning - commenting out 7/15
#for param in model.model.model.parameters():
 #  param.requires_grad = False  # Freeze all layers initially
 
# Training hyperparameters
model.train(
    data=yaml_file_path,
    epochs=100,
    imgsz=1024,
    batch=16,  # Adjust batch size based on GPU capacity
    #lr0=0.001,  # Initial learning rate
    #lrf=0.0001,  # Final learning rate (used for Cosine Annealing)
    #optimizer='AdamW',  # Use AdamW optimizer for better performance
    device=device,
    save_period=20,  # Save model checkpoint every 10 epochs
    patience=15,  # Early stopping if no improvement after 10 epochs
    #freeze=10, # Freeze first 10 layers - 
    #Augmentation
    augment=True,  # Enable data augmentation
    scale=0.5,
    degrees=180.0,
    #translate=0.2,
    #keep sifting minimal so corals aren't artificially cropped
    #Loss Weights
    box=10, #model focuses heavy on tight single boxes
    cls=1.5, #pushing model to commit to making classification of an object
    dfl = 1.5, #prevents 1-pixel "flat" edge box collapses (did not have this in V8.3, but including here as it shouldnt affect training, just will prevent flat-edge box collapses)
    #mosaic=True,  # Use mosaic augmentation
    #mixup=True,   # Use MixUp augmentation
    cos_lr=True,  # Cosine annealing learning rate
    seed=42,
    project=output_path,
    name='training_logs'  # TensorBoard logging directory
)

print("Training complete!")
